# dunnhumby "Breakfast at the Frat" weekly panel for ICDN

Step-by-step mapping of the observation unit

`(store, UPC, week) → (price, units, observed promo)`

We do **not** call `builder.run()` directly. Each cell invokes one method so we can inspect intermediates and freeze the sample without any lookahead.

**Files used:** `dunnhumby-transaction.csv` (524,950 real rows + blank trailing rows), `dunnhumby-products.csv` (58 UPCs), `dunnhumby-store.csv` (79 rows, 2 duplicate `STORE_ID`).

**Not reconstructed:** demand, prices. **Not inferred:** promotion — `feature`/`display`/`tpr_only` are observed, not a markdown proxy.

`week_id` is calendar-derived: `week_id_t = 1 + (date_t - date_1)/7d`. Missing calendar weeks would create a hole that is *not* compacted (empirically there are none here: 156 weeks, all exactly 7 days apart).

Selection (core stores, category, candidate SKUs) uses only `week_id <= 78`, the first half of the horizon, and is frozen before any model is fit.

In [16]:
from pathlib import Path
import sys

import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))
from dunn import CONCEPT_MAP, DunnConfig, DunnhumbyPanelBuilder

DATA_DIR = PROJECT_ROOT / "data" / "Dunnhumby"
OUT_DIR = DATA_DIR / "panel"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR    :", DATA_DIR)
print("Files       :", sorted(p.name for p in DATA_DIR.glob("*")))
pd.DataFrame({"source": list(CONCEPT_MAP), "dunnhumby_field": list(CONCEPT_MAP.values())})

PROJECT_ROOT: /home/thebigmonster/Github/nn-elasticity-additional-work
DATA_DIR    : /home/thebigmonster/Github/nn-elasticity-additional-work/data/Dunnhumby
Files       : ['dunnhumby-products.csv', 'dunnhumby-store.csv', 'dunnhumby-transaction.csv', 'panel']


,source,dunnhumby_field
0,store_num,store_code
1,upc,product_code
2,week_end_date,week_id (calendar-derived) + week_end_date (ke...
3,price,price
4,base_price,"base_price (audit: discount_depth, markdown_flag)"
5,units,units
6,feature|display|tpr_only,on_promo = 1[feature or display or tpr_only]; ...
7,spend|hhs|visits,master only (audit) -- never in the ICDN panel...
8,manufacturer,brand
9,sub_category,style


## 0. Config and builder

Selection thresholds are stored now but applied only on the first 78 weeks. `target_category` stays `None` until we inspect `category_stats` — it is never set from ICDN/MLP results.

In [17]:
config = DunnConfig(
    data_dir=DATA_DIR,
    out_dir=OUT_DIR,
    selection_frac=0.50,
    min_store_week_coverage=0.85,
    max_core_stores=None,
    min_observed_coverage=0.80,
    min_positive_rate=0.90,
    min_unique_prices=8,
    min_price_cv=0.03,
    min_core_store_presence=0.80,
    n_candidate_skus=20,
    n_skus=10,
    target_category=None,
)

builder = DunnhumbyPanelBuilder(config)

## 1. Raw loaders — transactions, products, stores

In [18]:
txn = builder.load_transactions()
print(txn.shape)
txn[["store_num", "upc", "week_end_date", "units", "price", "base_price", "feature", "display", "tpr_only"]].head()

transactions: dropped 157371 blank trailing rows; 524950 real rows remain
(524950, 15)


,store_num,upc,week_end_date,units,price,base_price,feature,display,tpr_only
0,367,1111009477,2009-01-14,13.0000,1.3900,1.5700,0,0,1
1,367,1111009497,2009-01-14,20.0000,1.3900,1.3900,0,0,0
2,367,1111009507,2009-01-14,14.0000,1.3800,1.3800,0,0,0
3,367,1111035398,2009-01-14,4.0000,3.5000,4.4900,0,0,1
4,367,1111038078,2009-01-14,3.0000,2.5000,2.5000,0,0,0


In [19]:
products = builder.load_products()
stores = builder.load_stores()
print(products.shape, stores.shape)
products.head()

products: 58 UPCs, categories=['BAG SNACKS', 'COLD CEREAL', 'FROZEN PIZZA', 'ORAL HYGIENE PRODUCTS']
store_id 4503: 2 duplicate rows differ only in ['SEG_VALUE_NAME']; keeping the first.
store_id 17627: 2 duplicate rows differ only in ['SEG_VALUE_NAME']; keeping the first.
stores: 77 unique store_code after dedup
(58, 7) (77, 4)


,product_code,category,brand,style,product_size_raw,size_value,size_unit
0,1111009477,BAG SNACKS,PRIVATE LABEL,PRETZELS,15 OZ,15.0000,OZ
1,1111009497,BAG SNACKS,PRIVATE LABEL,PRETZELS,15 OZ,15.0000,OZ
2,1111009507,BAG SNACKS,PRIVATE LABEL,PRETZELS,15 OZ,15.0000,OZ
3,1111035398,ORAL HYGIENE PRODUCTS,PRIVATE LABEL,MOUTHWASHES (ANTISEPTIC),1.5 LT,1.5000,LT
4,1111038078,ORAL HYGIENE PRODUCTS,PRIVATE LABEL,MOUTHWASHES (ANTISEPTIC),500 ML,500.0000,ML


## 2. Uniqueness — one row per (store, UPC, week)

In [20]:
builder.validate_uniqueness(txn)
print("OK: no duplicate store x UPC x week rows")

duplicate (store_num, upc, week_end_date) rows: 0
OK: no duplicate store x UPC x week rows


## 3. Calendar week_id (holes, if any, are not compacted)

In [21]:
txn = builder.build_week_index(txn)
print(txn["week_id"].min(), "→", txn["week_id"].max(), " n_weeks=", txn["week_id"].nunique())
txn[["week_end_date", "week_id"]].drop_duplicates().sort_values("week_id").head()

week_id spans 1..156 from 156 observed weeks (0 missing calendar weeks, NOT compacted)
1 → 156  n_weeks= 156


,week_end_date,week_id
0,2009-01-14,1
3158,2009-01-21,2
6288,2009-01-28,3
9422,2009-02-04,4
12545,2009-02-11,5


## 4. Merge product + store metadata

In [22]:
merged = builder.merge_metadata(txn, products, stores)
print(merged.shape)
merged[["store_code", "product_code", "category", "brand", "style", "product_size_raw", "size_value", "size_unit",
        "state", "store_segment", "sales_area_size"]].head()

(524950, 25)


,store_code,product_code,category,brand,style,product_size_raw,size_value,size_unit,state,store_segment,sales_area_size
0,367,1111009477,BAG SNACKS,PRIVATE LABEL,PRETZELS,15 OZ,15.0000,OZ,KY,VALUE,24721
1,367,1111009497,BAG SNACKS,PRIVATE LABEL,PRETZELS,15 OZ,15.0000,OZ,KY,VALUE,24721
2,367,1111009507,BAG SNACKS,PRIVATE LABEL,PRETZELS,15 OZ,15.0000,OZ,KY,VALUE,24721
3,367,1111035398,ORAL HYGIENE PRODUCTS,PRIVATE LABEL,MOUTHWASHES (ANTISEPTIC),1.5 LT,1.5000,LT,KY,VALUE,24721
4,367,1111038078,ORAL HYGIENE PRODUCTS,PRIVATE LABEL,MOUTHWASHES (ANTISEPTIC),500 ML,500.0000,ML,KY,VALUE,24721


## 5. Observed promo (feature/display/tpr) + audit-only diagnostics

In [23]:
merged = builder.compute_promo_and_audit(merged)
merged[["price", "base_price", "discount_depth", "markdown_flag", "price_value", "units_per_visit", "visits_per_hh", "on_promo"]].describe()

audit: price vs spend/units -> max abs diff = 8.881784197001252e-16  mean = 9.384286027156048e-17


,price,base_price,discount_depth,markdown_flag,price_value,units_per_visit,visits_per_hh,on_promo
count,"524,927.0000","524,765.0000","524,742.0000","524,742.0000","524,945.0000","524,950.0000","524,950.0000","524,950.0000"
mean,3.3822,3.6027,0.0572,0.2569,3.3821,1.1177,1.0228,0.2846
std,1.5593,1.6317,0.1143,0.4369,1.5594,0.2140,0.1323,0.4512
min,0.0000,0.5500,-1.3973,0.0000,0.0000,0.0000,1.0000,0.0000
25%,2.3600,2.5000,0.0000,0.0000,2.3600,1.0000,1.0000,0.0000
50%,2.9900,3.1700,0.0000,0.0000,2.9900,1.0385,1.0000,0.0000
75%,4.4900,4.5900,0.0347,1.0000,4.4900,1.1667,1.0000,1.0000
max,11.4600,11.4600,1.0000,1.0000,11.4600,25.5000,20.0000,1.0000


## 6. Assemble, validate and save the scientific master

In [24]:
master = builder.assemble_master(merged)
builder.validate_master(master)
master = builder.save_master(master)
builder.master = master
print(master.shape)
master.head()

rows=524950  zero-unit rows=5 (0.0010%)
missing price=23  non-positive price=1  (0.0046% of rows; manual reports <0.5% outliers)
Wrote /home/thebigmonster/Github/nn-elasticity-additional-work/data/Dunnhumby/panel/dunnhumby_weekly_master.parquet
(524950, 28)


,store_code,product_code,week_id,week_end_date,price,base_price,units,feature,display,tpr_only,on_promo,category,brand,style,product_size_raw,size_value,size_unit,state,store_segment,sales_area_size,spend,hhs,visits,discount_depth,units_per_visit,visits_per_hh,markdown_flag,price_value
0,10019,1111009477,1,2009-01-14,1.1700,1.1800,14.0000,0,1,0,1,BAG SNACKS,PRIVATE LABEL,PRETZELS,15 OZ,15.0000,OZ,TX,VALUE,"35,675.0000",16.3800,13.0000,13.0000,0.0085,1.0769,1.0000,1.0000,1.1700
1,10019,1111009497,1,2009-01-14,1.1700,1.1800,25.0000,0,1,0,1,BAG SNACKS,PRIVATE LABEL,PRETZELS,15 OZ,15.0000,OZ,TX,VALUE,"35,675.0000",29.2500,22.0000,24.0000,0.0085,1.0417,1.0909,1.0000,1.1700
2,10019,1111009507,1,2009-01-14,1.1800,1.1800,8.0000,0,0,0,0,BAG SNACKS,PRIVATE LABEL,PRETZELS,15 OZ,15.0000,OZ,TX,VALUE,"35,675.0000",9.4400,8.0000,8.0000,0.0000,1.0000,1.0000,0.0000,1.1800
3,10019,1111038078,1,2009-01-14,1.1000,1.4000,4.0000,0,0,1,1,ORAL HYGIENE PRODUCTS,PRIVATE LABEL,MOUTHWASHES (ANTISEPTIC),500 ML,500.0000,ML,TX,VALUE,"35,675.0000",4.4000,3.0000,3.0000,0.2143,1.3333,1.0000,1.0000,1.1000
4,10019,1111038080,1,2009-01-14,1.2500,1.5000,4.0000,0,0,1,1,ORAL HYGIENE PRODUCTS,PRIVATE LABEL,MOUTHWASHES (ANTISEPTIC),500 ML,500.0000,ML,TX,VALUE,"35,675.0000",5.0000,4.0000,4.0000,0.1667,1.0000,1.0000,1.0000,1.2500


## 7. Selection window (weeks 1–78, no lookahead)

Store, category and SKU choices below use only this window and are frozen afterwards.

In [25]:
selection = builder.selection_sample()
print(selection.shape)

selection window: week_id <= 78 (78 of 156 weeks)
(256679, 28)


## 8. Core stores (overall activity, not category-specific)

In [26]:
core_stores = builder.select_core_stores(selection)
print(len(core_stores), "core stores")
builder.store_stats.sort_values("week_coverage", ascending=False).head(10)

core stores: 76 of 77 pass week_coverage >= 0.85
Wrote /home/thebigmonster/Github/nn-elasticity-additional-work/data/Dunnhumby/panel/dunnhumby_store_diagnostics.csv
76 core stores


,store_code,n_obs,n_weeks,n_products,total_units,week_coverage
0,10019,2742,78,45,"39,179.0000",1.0000
1,11757,3575,78,54,"80,901.0000",1.0000
2,11761,3778,78,54,"84,268.0000",1.0000
3,11967,2528,78,44,"30,782.0000",1.0000
4,11993,3408,78,47,"76,617.0000",1.0000
5,12011,3056,78,47,"44,252.0000",1.0000
6,13609,3845,78,54,"97,852.0000",1.0000
7,13827,3202,78,48,"57,766.0000",1.0000
8,13837,3200,78,47,"59,479.0000",1.0000
9,13853,3203,78,49,"42,299.0000",1.0000


## 9. Product diagnostics (core stores, selection window)

In [27]:
selection_core = selection[selection["store_code"].isin(core_stores)]
product_stats = builder.compute_product_stats(selection_core)
product_stats.sort_values("coverage", ascending=False).head(15)

Wrote /home/thebigmonster/Github/nn-elasticity-additional-work/data/Dunnhumby/panel/dunnhumby_product_diagnostics.csv


,product_code,category,brand,style,product_size_raw,size_value,size_unit,n_obs,positive_obs,n_stores,global_unique_prices,mean_price,std_price,total_units,promo_rate,coverage,positive_rate,global_price_cv,store_presence,median_within_store_cv,share_stores_with_variation
14,1600027564,COLD CEREAL,GENERAL MI,ALL FAMILY CEREAL,12 OZ,12.0000,OZ,5927,5927,76,235,2.6940,0.4428,"241,590.0000",0.4862,0.9998,1.0000,0.1644,1.0000,0.1575,1.0000
7,1111085345,COLD CEREAL,PRIVATE LABEL,ADULT CEREAL,20 OZ,20.0000,OZ,5926,5926,76,48,1.7597,0.1091,"187,608.0000",0.2457,0.9997,1.0000,0.0620,1.0000,0.0609,1.0000
12,1600027527,COLD CEREAL,GENERAL MI,ALL FAMILY CEREAL,12.25 OZ,12.2500,OZ,5926,5926,76,155,2.8370,0.3868,"381,461.0000",0.1902,0.9997,1.0000,0.1364,1.0000,0.1355,1.0000
22,3000006560,COLD CEREAL,QUAKER,KIDS CEREAL,13 OZ,13.0000,OZ,5926,5926,76,70,2.4982,0.2087,"184,173.0000",0.1873,0.9997,1.0000,0.0835,1.0000,0.0834,1.0000
13,1600027528,COLD CEREAL,GENERAL MI,ALL FAMILY CEREAL,18 OZ,18.0000,OZ,5924,5924,76,181,4.1873,0.5855,"186,901.0000",0.1175,0.9993,1.0000,0.1398,1.0000,0.1332,1.0000
32,3800031838,COLD CEREAL,KELLOGG,KIDS CEREAL,15 OZ,15.0000,OZ,5922,5922,76,188,2.7318,0.3635,"306,625.0000",0.3269,0.9990,1.0000,0.1331,1.0000,0.1325,1.0000
8,1111085350,COLD CEREAL,PRIVATE LABEL,ALL FAMILY CEREAL,18 OZ,18.0000,OZ,5919,5919,76,73,2.1505,0.1796,"224,415.0000",0.3203,0.9985,1.0000,0.0835,1.0000,0.0823,1.0000
6,1111085319,COLD CEREAL,PRIVATE LABEL,ALL FAMILY CEREAL,12.25 OZ,12.2500,OZ,5919,5919,76,57,1.7632,0.1118,"126,689.0000",0.2321,0.9985,1.0000,0.0634,1.0000,0.0622,1.0000
23,3000006610,COLD CEREAL,QUAKER,KIDS CEREAL,14 OZ,14.0000,OZ,5914,5914,76,78,2.4929,0.2140,"121,260.0000",0.2090,0.9976,1.0000,0.0858,1.0000,0.0837,1.0000
1,1111009497,BAG SNACKS,PRIVATE LABEL,PRETZELS,15 OZ,15.0000,OZ,5912,5912,76,66,1.2576,0.1957,"289,107.0000",0.4427,0.9973,1.0000,0.1556,1.0000,0.0890,1.0000


## 10. Density / price-variation screen

In [28]:
eligible = builder.screen_eligible(product_stats)
print(eligible.groupby("category", observed=True).size())
eligible[["product_code", "category", "brand", "style", "product_size_raw",
          "size_value", "size_unit", "coverage", "positive_rate",
          "share_stores_with_variation", "median_within_store_cv",
          "store_presence"]].head(20)

eligible SKUs: 36 of 55
category
BAG SNACKS                8
COLD CEREAL              12
FROZEN PIZZA              8
ORAL HYGIENE PRODUCTS     8
dtype: int64


,product_code,category,brand,style,product_size_raw,size_value,size_unit,coverage,positive_rate,share_stores_with_variation,median_within_store_cv,store_presence
14,1600027564,COLD CEREAL,GENERAL MI,ALL FAMILY CEREAL,12 OZ,12.0000,OZ,0.9998,1.0000,1.0000,0.1575,1.0000
12,1600027527,COLD CEREAL,GENERAL MI,ALL FAMILY CEREAL,12.25 OZ,12.2500,OZ,0.9997,1.0000,1.0000,0.1355,1.0000
22,3000006560,COLD CEREAL,QUAKER,KIDS CEREAL,13 OZ,13.0000,OZ,0.9997,1.0000,1.0000,0.0834,1.0000
7,1111085345,COLD CEREAL,PRIVATE LABEL,ADULT CEREAL,20 OZ,20.0000,OZ,0.9997,1.0000,1.0000,0.0609,1.0000
13,1600027528,COLD CEREAL,GENERAL MI,ALL FAMILY CEREAL,18 OZ,18.0000,OZ,0.9993,1.0000,1.0000,0.1332,1.0000
32,3800031838,COLD CEREAL,KELLOGG,KIDS CEREAL,15 OZ,15.0000,OZ,0.9990,1.0000,1.0000,0.1325,1.0000
8,1111085350,COLD CEREAL,PRIVATE LABEL,ALL FAMILY CEREAL,18 OZ,18.0000,OZ,0.9985,1.0000,1.0000,0.0823,1.0000
6,1111085319,COLD CEREAL,PRIVATE LABEL,ALL FAMILY CEREAL,12.25 OZ,12.2500,OZ,0.9985,1.0000,1.0000,0.0622,1.0000
23,3000006610,COLD CEREAL,QUAKER,KIDS CEREAL,14 OZ,14.0000,OZ,0.9976,1.0000,1.0000,0.0837,1.0000
33,3800039118,COLD CEREAL,KELLOGG,KIDS CEREAL,12.2 OZ,12.2000,OZ,0.9973,1.0000,1.0000,0.1308,1.0000


## 11. Rank categories — STOP and inspect

The category with the most eligible SKUs wins; ties would be broken by joint store-week co-occurrence. No model output is used here.

In [29]:
category_stats = builder.diagnose_selection_window.__wrapped__ if False else None  # placeholder, not used
category_stats = builder.rank_categories(eligible)
category_stats

Wrote /home/thebigmonster/Github/nn-elasticity-additional-work/data/Dunnhumby/panel/dunnhumby_category_diagnostics.csv
             category  n_eligible  median_coverage  median_within_store_cv  median_store_presence    total_units
          COLD CEREAL          12           0.9987                  0.1044                 1.0000 2,492,596.0000
         FROZEN PIZZA           8           0.9717                  0.1226                 1.0000   597,313.0000
           BAG SNACKS           8           0.9324                  0.0848                 1.0000 1,072,028.0000
ORAL HYGIENE PRODUCTS           8           0.8610                  0.1190                 1.0000   213,033.0000
[WARN] category 'FROZEN PIZZA': 8 eligible SKUs < 10 required. freeze_universe will raise if you choose this category.
[WARN] category 'BAG SNACKS': 8 eligible SKUs < 10 required. freeze_universe will raise if you choose this category.
[WARN] category 'ORAL HYGIENE PRODUCTS': 8 eligible SKUs < 10 required. freeze_u

,category,n_eligible,median_coverage,median_within_store_cv,median_store_presence,total_units
1,COLD CEREAL,12,0.9987,0.1044,1.0000,"2,492,596.0000"
2,FROZEN PIZZA,8,0.9717,0.1226,1.0000,"597,313.0000"
0,BAG SNACKS,8,0.9324,0.0848,1.0000,"1,072,028.0000"
3,ORAL HYGIENE PRODUCTS,8,0.8610,0.1190,1.0000,"213,033.0000"


## 12. Freeze category + Jaccard co-occurrence SKU selection

Set **after** inspecting `category_stats`, never from ICDN vs MLP results.

In [30]:
TARGET_CATEGORY = str(category_stats.iloc[0]["category"])
print("Using category =", TARGET_CATEGORY)

selection = builder.freeze_universe(TARGET_CATEGORY)
selection.to_dict()

Using category = COLD CEREAL
SELECTED_PRODUCTS: ['1600027564', '1600027527', '3000006560', '1111085345', '1600027528', '3800031838', '1111085350', '1111085319', '3000006610', '3800039118']
store-weeks with >= 8/10 SKUs jointly observed: 0.9997
store-weeks with all 10/10 SKUs jointly observed: 0.9899
Wrote frozen store/SKU manifest under /home/thebigmonster/Github/nn-elasticity-additional-work/data/Dunnhumby/panel


{'cutoff_week_id': 78,
 'category': 'COLD CEREAL',
 'stores': ['2277',
  '24991',
  '25027',
  '2281',
  '13609',
  '9825',
  '21227',
  '28909',
  '6179',
  '26973',
  '389',
  '21237',
  '11761',
  '4245',
  '23075',
  '19265',
  '15547',
  '23061',
  '8041',
  '21213',
  '15541',
  '26981',
  '2279',
  '11757',
  '25021',
  '25001',
  '2513',
  '21221',
  '23067',
  '6187',
  '6379',
  '15531',
  '11993',
  '613',
  '25229',
  '4259',
  '4503',
  '4489',
  '15765',
  '17627',
  '23345',
  '26983',
  '8263',
  '623',
  '29159',
  '19533',
  '13859',
  '27175',
  '19521',
  '2495',
  '13853',
  '13827',
  '13837',
  '25261',
  '23327',
  '15763',
  '2505',
  '17615',
  '21479',
  '19523',
  '25233',
  '12011',
  '6431',
  '4521',
  '23349',
  '21485',
  '17599',
  '367',
  '25253',
  '2541',
  '10019',
  '15755',
  '2523',
  '11967',
  '23055',
  '8035'],
 'candidate_products': ['1600027564',
  '1600027527',
  '3000006560',
  '1111085345',
  '1600027528',
  '3800031838',
  '1111085350

In [31]:
sku_sizes = (
    builder.master[builder.master["product_code"].isin(selection.product_codes)]
    [["product_code", "product_size_raw", "size_value", "size_unit"]]
    .drop_duplicates()
    .sort_values("product_code")
)
print(sku_sizes)
assert sku_sizes["size_unit"].nunique() == 1
assert sku_sizes["size_value"].notna().all()

   product_code product_size_raw  size_value size_unit
5    1111085319         12.25 OZ     12.2500        OZ
6    1111085345            20 OZ     20.0000        OZ
7    1111085350            18 OZ     18.0000        OZ
11   1600027527         12.25 OZ     12.2500        OZ
12   1600027528            18 OZ     18.0000        OZ
13   1600027564            12 OZ     12.0000        OZ
17   3000006560            13 OZ     13.0000        OZ
18   3000006610            14 OZ     14.0000        OZ
26   3800031838            15 OZ     15.0000        OZ
27   3800039118          12.2 OZ     12.2000        OZ


## 13. ICDN panel — full horizon, frozen stores/SKUs, units>0 & price>0

In [32]:
icdn_panel = builder.build_icdn_panel()
print(icdn_panel.shape)
print(icdn_panel.dtypes)
icdn_panel.head()

dropping 1 zero-unit rows and 2 missing/non-positive price rows for ICDN (never recoded as positive)
size: numeric OZ, values=[np.float64(12.0), np.float64(12.2), np.float64(12.25), np.float64(13.0), np.float64(14.0), np.float64(15.0), np.float64(18.0), np.float64(20.0)]
Wrote /home/thebigmonster/Github/nn-elasticity-additional-work/data/Dunnhumby/panel/dunnhumby_icdn_panel.parquet
              n_obs  n_weeks  n_stores  mean_units
product_code                                      
1111085319    11828      156        76     21.6381
1111085345    11849      156        76     30.5485
1111085350    11832      156        76     38.6881
1600027527    11854      156        76     66.7679
1600027528    11852      156        76     30.5605
1600027564    11849      156        76     42.7838
3000006560    10071      152        76     32.9763
3000006610    10386      156        76     23.8876
3800031838    11849      156        76     47.2698
3800039118    11828      156        76     38.4303
(11

,store_code,product_code,week_id,price,units,on_promo,category,brand,style,size
0,10019,1111085319,1,1.8300,9.0000,0,COLD CEREAL,PRIVATE LABEL,ALL FAMILY CEREAL,12.2500
1,10019,1111085345,1,1.7600,12.0000,0,COLD CEREAL,PRIVATE LABEL,ADULT CEREAL,20.0000
2,10019,1111085350,1,1.8900,5.0000,1,COLD CEREAL,PRIVATE LABEL,ALL FAMILY CEREAL,18.0000
3,10019,1600027527,1,3.0600,24.0000,0,COLD CEREAL,GENERAL MI,ALL FAMILY CEREAL,12.2500
4,10019,1600027528,1,4.1500,6.0000,0,COLD CEREAL,GENERAL MI,ALL FAMILY CEREAL,18.0000


## 14. Final checks

In [33]:
assert (icdn_panel["price"] > 0).all()
assert (icdn_panel["units"] > 0).all()
assert set(icdn_panel["on_promo"].unique()).issubset({0, 1})
assert not icdn_panel.duplicated(["store_code", "product_code", "week_id"]).any()
assert not {"spend", "hhs", "visits"} & set(icdn_panel.columns)

print(
    "n_stores", icdn_panel["store_code"].nunique(),
    "n_skus", icdn_panel["product_code"].nunique(),
    "n_weeks", icdn_panel["week_id"].nunique(),
    "promo_rate", float(icdn_panel["on_promo"].mean()),
)
print(OUT_DIR)
print(sorted(p.name for p in OUT_DIR.glob("*") if p.suffix in {".parquet", ".csv", ".json"}))

n_stores 76 n_skus 10 n_weeks 156 promo_rate 0.25965728571676594
/home/thebigmonster/Github/nn-elasticity-additional-work/data/Dunnhumby/panel
['dunnhumby_category_diagnostics.csv', 'dunnhumby_icdn_panel.parquet', 'dunnhumby_product_diagnostics.csv', 'dunnhumby_selected_products.csv', 'dunnhumby_selected_stores.csv', 'dunnhumby_selection.json', 'dunnhumby_store_diagnostics.csv', 'dunnhumby_weekly_master.parquet']
